# NZ STR paper — one-notebook pipeline (v2, point-in-time conservative)
Chạy lần lượt từng ô từ trên xuống (**Shift+Enter**). Mọi kết quả được tính từ `01_raw` và ghi vào thư mục `OUT_DIR`.
Notebook này **không phụ thuộc Claude** — chỉ cần Python + các thư viện trong `09_pipeline/requirements.txt`.

## 0. Đường dẫn dự án — SỬA DÒNG `PROJECT_ROOT` cho đúng máy của bạn

In [ ]:
from pathlib import Path
import os, sys

# Thư mục chứa 01_raw/, 04_code/, 09_pipeline/ ...
PROJECT_ROOT = Path(r"C:\Users\yxt2919\7th_NZ_STR_Construction_AI")   # <-- SỬA NẾU KHÁC
OUT_DIR      = "09_pipeline_out_nb"   # thư mục kết quả riêng của notebook (không đụng file gốc)

if not (PROJECT_ROOT / "01_raw").exists():
    # thử tự tìm: notebook đặt trong 09_pipeline/ hoặc ở thư mục gốc
    for cand in [Path.cwd(), Path.cwd().parent]:
        if (cand / "01_raw").exists():
            PROJECT_ROOT = cand
assert (PROJECT_ROOT / "01_raw").exists(), f"Không thấy 01_raw trong {PROJECT_ROOT} — sửa PROJECT_ROOT"
os.chdir(PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)
print(sorted(p.name for p in PROJECT_ROOT.iterdir()))

## 1. Cài thư viện (chỉ cần chạy lần đầu; sau đó Kernel → Restart)

In [ ]:
import subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "--user", "-q", "-r",
                str(PROJECT_ROOT / "09_pipeline" / "requirements.txt")], check=False)
print("Xong. Nếu vừa cài mới, chọn Kernel > Restart rồi chạy lại từ ô 0.")

## 2. Nạp pipeline và cấu hình

In [ ]:
import importlib.util, yaml, pandas as pd, numpy as np, time
pd.set_option("display.max_columns", 60, "display.width", 200)

spec = importlib.util.spec_from_file_location("run_all", PROJECT_ROOT / "09_pipeline" / "run_all.py")
ra = importlib.util.module_from_spec(spec); spec.loader.exec_module(ra)

CFG = yaml.safe_load(open(PROJECT_ROOT / "09_pipeline" / "config.yaml", encoding="utf-8"))
CFG["paths"]["out_dir"] = OUT_DIR
ra.CFG = CFG
ra.OUT = Path(OUT_DIR); ra.OUT.mkdir(parents=True, exist_ok=True)
OUT = ra.OUT
print(yaml.safe_dump(CFG["paths"], allow_unicode=True))

## 3. Kiểm tra file raw có đúng tên/đúng chỗ không
Nếu một file báo **MISSING**, sửa đường dẫn trong `CFG["paths"]` ở ô dưới (tên file thật có thể khác, ví dụ `.csv` vs `.xlsx`).

In [ ]:
for k, v in CFG["paths"].items():
    if k in ("out_dir", "consents_tidy") or v is None: continue
    print(f"{'OK     ' if Path(v).exists() else 'MISSING'} {k}: {v}")
print("\nSnapshots:")
for s in ra.snapshots():
    files = sorted(p.name for p in s.iterdir())
    print(" ", s.name, files)

In [ ]:
# Nếu tên file thật khác, sửa ở đây rồi chạy lại ô trên. Ví dụ:
# CFG["paths"]["consents_file"] = "01_raw/construction/statsnz_building_consents_ta_monthly_2020M01_2026M07.xlsx"
# CFG["paths"]["adp_file"]      = "01_raw/seasonality/ADP_All_Measures.csv"
ra.CFG = CFG

## 4. Xem nhanh định dạng 3 file context (để chắc loader đọc đúng)
- **Consents**: cần thấy các dòng tiêu đề Infoshare rồi các dòng `2020M01`, `2020M02`…
- **Bond**: cần có cột thời gian, `Location`, và `Geometric Mean Rent` (hoặc `Median Rent`).
- **ADP**: cần có cột thời gian + cột giá trị; nếu có nhiều measure/khu vực, đặt `CFG["seasonality"]["filters"]`.

In [ ]:
print("=== CONSENTS (12 dòng đầu) ===")
display(ra.read_csv_any(CFG["paths"]["consents_file"], header=None, dtype=str, nrows=12).iloc[:, :8])
print("=== BOND ===")
b = ra.read_csv_any(CFG["paths"]["bond_file"], nrows=5); display(b)
print("=== ADP ===")
a = ra.read_csv_any(CFG["paths"]["adp_file"]); print(a.columns.tolist()); display(a.head())
for c in a.columns:
    if a[c].dtype == object and a[c].nunique() < 30:
        print(c, "->", a[c].unique()[:30])

In [ ]:
# Chỉnh nếu cần sau khi xem ô trên, ví dụ:
# CFG["seasonality"]["time_col"]  = "Period"
# CFG["seasonality"]["value_col"] = "Value"
# CFG["seasonality"]["filters"]   = {"Measure": "Guest nights", "Region": "Total New Zealand"}
# CFG["consents"]["include_regex"] = "new"
# CFG["consents"]["exclude_regex"] = "total|value|floor area|\\$"
ra.CFG = CFG

## 5. (Khuyến nghị) Lịch phát hành chính thức của Building Consents
Ô dưới tạo file mẫu `07_provenance/consents_release_calendar.csv`. Mở file bằng Excel, điền cột `release_date` (YYYY-MM-DD) và `source_url`
từ trang *Building consents issued: <tháng>* của Stats NZ. **Không đoán.** Nếu bỏ trống, pipeline dùng quy tắc thận trọng M−3 và ghi *UNVERIFIED*.

In [ ]:
cal = Path(CFG["paths"]["consents_release_calendar"])
if not cal.exists():
    cal.parent.mkdir(parents=True, exist_ok=True)
    months = pd.period_range("2025-01", "2026-07", freq="M").astype(str)
    pd.DataFrame({"ref_month": months, "release_date": "", "source_url": ""}).to_csv(cal, index=False)
    print("Đã tạo mẫu:", cal, "— điền release_date rồi chạy tiếp")
else:
    display(pd.read_csv(cal))

## 6. Chạy các stage (mỗi ô một stage; có thể chạy lại riêng từng ô)

In [ ]:
def run(stage_fn, name):
    t = time.time(); ra.log(f"stage {name}: start"); stage_fn(); ra.log(f"stage {name}: {time.time()-t:.0f}s")

run(ra.stage_manifest, "manifest")      # băm SHA-256 mọi file raw (vài phút vì reviews lớn)
display(pd.read_csv(OUT / "SNAPSHOT_MANIFEST.csv").head(15))

In [ ]:
run(ra.stage_panel, "panel")            # panel listing-origin, nhãn, TA join, audit
display(pd.read_csv(OUT / "audit_snapshots.csv"))
display(pd.read_csv(OUT / "audit_origins.csv"))

In [ ]:
run(ra.stage_features, "features")      # calendar + reviews + context + embedding review (lần đầu ~30–90 phút, có cache)
for f in ["audit_calendar.csv", "audit_reviews.csv", "audit_construction_timing.csv",
          "audit_rental_timing.csv", "audit_consents_columns.csv", "audit_bond_unmatched.csv"]:
    p = OUT / f
    if p.exists(): print("===", f); display(pd.read_csv(p).head(80))

### ✅ Kiểm tra trước khi chạy mô hình
1. `audit_consents_columns.csv`: chỉ các cột *new dwellings / number* có `used=True`, đủ ~67 TA.
2. `audit_bond_unmatched.csv`: chỉ các dòng không phải TA (tổng, v.v.).
3. `audit_origins.csv`: `ta_unmatched` ≈ 0.
4. Log không có `WARNING ... DROPPED`.
Nếu sai: sửa `CFG` ở ô 4, rồi chạy lại **từ ô stage features**.

In [ ]:
X = pd.read_parquet(OUT / "features.parquet")
fam = ra.families(X)
miss = pd.DataFrame({f: X[c].isna().mean() for f, cols in fam.items() for c in cols}, index=["missing_share"]).T
print({k: len(v) for k, v in fam.items()})
display(miss.sort_values("missing_share", ascending=False).head(25))

## 7. Mô hình: dev T4–T9 → đóng băng → đánh giá T10 một lần

In [ ]:
run(ra.stage_models, "models")          # ~90 lần fit (XGBoost, logistic, MLP); ~1–3 giờ tùy máy
import json
print(json.dumps(json.load(open(OUT / "FROZEN_SPEC.json")), indent=2)[:1500])

## 8. Kết quả

In [ ]:
R = json.load(open(OUT / "RESULTS.json"))
print("Selected:", R["selected"], "| predictors:", R["n_predictors"], "| threshold:", round(R["threshold"], 6), "| calibration:", R["calibration"])
print("T10 raw:", {k: round(v, 4) if isinstance(v, float) else v for k, v in R["t10_raw"].items()})
print("T10 CI low :", R["t10_ci_low"]); print("T10 CI high:", R["t10_ci_high"])
print("Confusion:", R["confusion"]); print("Secondary:", R["secondary"])
display(pd.read_csv(OUT / "dev_summary.csv").sort_values("pr_auc", ascending=False))
display(pd.read_csv(OUT / "T10_topk.csv"))
display(pd.read_csv(OUT / "T10_deciles.csv"))

In [ ]:
# So sánh với số frozen v1 trong manuscript cũ
v1 = {"T10 ROC-AUC": 0.743199, "T10 PR-AUC": 0.069483, "T10 raw Brier": 0.021475,
      "Dev macro ROC-AUC": 0.738356, "Dev macro PR-AUC": 0.074924, "T10 N": 51096, "T10 events": 1147,
      "Origins": 491140, "Price missing %": 35.20, "Scraped later %": 98.48, "Reappear T+2 %": 12.76}
sel = [d for d in R["dev"] if f"{d['model']} {d['set']}" == R["selected"]][0]
v2 = {"T10 ROC-AUC": R["t10_raw"]["roc_auc"], "T10 PR-AUC": R["t10_raw"]["pr_auc"], "T10 raw Brier": R["t10_raw"]["brier"],
      "Dev macro ROC-AUC": sel["roc_auc"], "Dev macro PR-AUC": sel["pr_auc"], "T10 N": R["t10_raw"]["n"],
      "T10 events": R["t10_raw"]["events"], "Origins": R["n_origins"], "Price missing %": R["price_missing_pct"],
      "Scraped later %": R["scraped_later_pct"], "Reappear T+2 %": 100 * R["reappear"] / max(R["reappear_den"], 1)}
cmp = pd.DataFrame({"v1_frozen": v1, "v2_point_in_time": v2}); cmp["diff"] = cmp.v2_point_in_time - cmp.v1_frozen
cmp.to_csv(OUT / "v1_vs_v2.csv"); display(cmp)

In [ ]:
from IPython.display import IFrame
print(open(OUT / "results_macros.tex", encoding="utf-8").read())
print("Hình:", [p.name for p in (OUT / "figures").glob("*.pdf")])

## 9. Đưa vào manuscript
Chép `results_macros.tex` và thư mục `figures/` vào cạnh `main.tex`, thêm `\input{results_macros}` vào preamble,
rồi thay số cứng bằng macro, ví dụ `ROC-AUC \ResTtenROC{} (95\% CI \ResTtenROClo--\ResTtenROChi)`.